## Peach text-generation pipeline (Chinese game UGC)

This notebook mirrors the Streamlit app in `app/app.py`: same `model_utils.py` helpers, lazy loading, and JSON export.

**Usage**

1. Use a **GPU runtime** when possible (9B model; CPU may be very slow or run out of memory).
2. Run all cells from top to bottom (**Restart & Run All**).
3. Edit the **Settings** cell (character / world fields; `USER_PROMPT` is built with `model_utils.build_narrative_user_prompt`, same as the Streamlit app), plus decoding parameters and `RUN_GENERATION`.
4. Exported JSON is written under `exports/` at the repository root.

**Course alignment**: demonstrates the **data → model → application** link for the generation leg of the project (see `docs/02_text_generation.md`). The safety **classification** pipeline is added in a later milestone.


## Settings cell

Edit **structured narrative fields** (`CHAR_A`, `CHAR_B`, `ENV`, `GEN_REQUIREMENTS`, optional `PLOT_BEATS`); `USER_PROMPT` is built via `model_utils.build_narrative_user_prompt` (same function as the Streamlit app’s narrative mode). Adjust decoding parameters and `RUN_GENERATION` here (no `input()` — suitable for **Restart & Run All**).


In [ ]:
import sys
from pathlib import Path

# Same import path as the cell below so this cell works on **Restart & Run All**.
_REPO_ROOT = Path("..").resolve()
_APP_DIR = _REPO_ROOT / "app"
if str(_APP_DIR) not in sys.path:
    sys.path.insert(0, str(_APP_DIR))

from model_utils import build_narrative_user_prompt  # noqa: E402

# Structured narrative inputs — edit CHAR_A, CHAR_B, ENV, GEN_REQUIREMENTS, and optionally PLOT_BEATS.
# USER_PROMPT is assembled via model_utils.build_narrative_user_prompt (shared with app/app.py).

CHAR_A = {
    "name": "艾德莉娅",
    "identity": "被放逐的精灵游侠，独来独往，擅长弓箭",
    "personality": "外冷内热，对陌生人充满戒备，但内心渴望找到失散的族人，说话简短直接，不喜欢绕弯子",
    "state": "身受轻伤，正在篝火旁处理伤口，处于疲惫状态",
}

CHAR_B = {
    "name": "罗根",
    "identity": "流浪骑士，曾是王国军团的队长，现在为了寻找解药而游历",
    "personality": "正直、略显话痨，喜欢用逻辑分析问题，有强烈的正义感",
    "state": "刚刚从一场魔物袭击中救下了角色 A，自己也消耗了大量体力",
}

ENV = {
    "place": "幽暗的“叹息森林”深处，一棵巨大的古树根形成的天然洞穴内",
    "atmosphere": "潮湿、阴冷，洞外下着大雨，远处偶尔传来狼嚎声，篝火是唯一的光源",
}

PLOT_BEATS = [
    "开场：A 对 B 的救助不领情，怀疑 B 是追兵，气氛紧张。",
    "转折：B 为消除误会说出在找“星尘花”解药，与 A 寻族人的线索相交。",
    "高潮：发现目标都指向废弃的“灰烬神殿”，伤病下单独前往危险，两人暂时结盟。",
    "结尾：约定明早出发，暗示神殿内有更大危险。",
]

GEN_REQUIREMENTS = {
    "format": "连续正文；用“姓名：台词”写对话，动作与环境用普通句子插入，不要再写小标题或目录。",
    "style": "西式奇幻 RPG，口语自然，简洁有画面感。",
    "length": "约 350-450 字（一段或少数段，不要分很多节）。",
}

USER_PROMPT = build_narrative_user_prompt(
    char_a=CHAR_A,
    char_b=CHAR_B,
    env=ENV,
    plot_beats=PLOT_BEATS,
    gen_requirements=GEN_REQUIREMENTS,
)

MAX_NEW_TOKENS = 512
TEMPERATURE = 0.5
TOP_P = 0.7
DO_SAMPLE = True
USE_CHINESE_ROLEPLAY_WRAP = False

# Optional: override narrative system prompt (empty = DEFAULT_NARRATIVE_SYSTEM_PROMPT_ZH, same as Streamlit narrative mode)
CUSTOM_SYSTEM_PROMPT = ""

# Set False to skip model download / inference (keeps Restart & Run All lightweight)
RUN_GENERATION = True


## Environment / dependencies

Install packages once per session; long pip output is hidden with `%%capture`.


In [ ]:
%%capture
# Install dependencies (same family as app/requirements.txt)
!pip install "transformers>=4.40.0,<5.0.0" "torch>=2.2.0,<3.0" sentencepiece safetensors accelerate streamlit --quiet

import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
from transformers import logging as hf_logging

hf_logging.set_verbosity_error()


## Imports and path to `app/model_utils.py`

The notebook lives in `notebooks/`; we add the sibling `app/` folder to `sys.path` so imports match the Streamlit app.


In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path("..").resolve()
APP_DIR = REPO_ROOT / "app"
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

from model_utils import (  # noqa: E402
    MAX_PROMPT_CHARS,
    TEXT_GEN_MODEL_ID,
    DEFAULT_NARRATIVE_SYSTEM_PROMPT_ZH,
    build_generation_export_dict,
    build_narrative_user_prompt,
    generate_ugc_text,
    get_text_generation_pipeline,
    normalize_and_truncate_prompt,
    resolve_eos_token_id,
    save_generation_json,
)

print("REPO_ROOT:", REPO_ROOT)
print("Model:", TEXT_GEN_MODEL_ID)
print("Max prompt chars:", MAX_PROMPT_CHARS)


## Load model, generate, export JSON

On failure (network, CUDA OOM, etc.) the cell prints a clear message so **Restart & Run All** still finishes if you set `RUN_GENERATION = False` above.


In [ ]:
if not RUN_GENERATION:
    print("RUN_GENERATION is False — skipped model load and inference.")
else:
    try:
        trimmed, truncated = normalize_and_truncate_prompt(USER_PROMPT)
    except ValueError as exc:
        print("Invalid prompt:", exc)
        raise

    if truncated:
        print(
            f"Warning: prompt truncated to {len(trimmed)} characters (max {MAX_PROMPT_CHARS})."
        )

    system_prompt = (
        CUSTOM_SYSTEM_PROMPT.strip() or DEFAULT_NARRATIVE_SYSTEM_PROMPT_ZH
    )

    try:
        print("Loading model (first run may download weights)...")
        pipe = get_text_generation_pipeline()
        print("Load complete.")
    except RuntimeError as exc:
        print("Model load failed:", exc)
        print("Tip: use a GPU runtime or set RUN_GENERATION = False for a dry run.")
    else:
        gen_params = {
            "max_new_tokens": MAX_NEW_TOKENS,
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "do_sample": DO_SAMPLE,
            "repetition_penalty": 1.15,
            "no_repeat_ngram_size": 3,
            "eos_token_id": resolve_eos_token_id(pipe.tokenizer),
            "use_chinese_roleplay_wrap": USE_CHINESE_ROLEPLAY_WRAP,
        }
        try:
            generated_text, elapsed_sec, messages = generate_ugc_text(
                pipe,
                trimmed,
                system_prompt=system_prompt,
                use_chinese_roleplay_wrap=USE_CHINESE_ROLEPLAY_WRAP,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                do_sample=DO_SAMPLE,
                repetition_penalty=1.15,
                no_repeat_ngram_size=3,
            )
        except Exception as exc:
            print("Generation failed:", exc)
        else:
            print(f"Elapsed: {elapsed_sec:.2f} s")
            print("--- generated_text ---")
            print(generated_text)

            export = build_generation_export_dict(
                generated_text=generated_text,
                messages=messages,
                generation_params=gen_params,
                elapsed_sec=elapsed_sec,
                model_id=TEXT_GEN_MODEL_ID,
            )

            export_dir = REPO_ROOT / "exports"
            export_dir.mkdir(parents=True, exist_ok=True)
            out_path = export_dir / "peach_generation_notebook.json"
            save_generation_json(export, str(out_path))
            print("Saved JSON to:", out_path)
            print(json.dumps(export, ensure_ascii=False, indent=2)[:2000], "...")
